In [6]:
import pandas as pd
import difflib
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
def extract_features():
    df = pd.read_csv("Data/ast_dataset.csv")

    df = df.dropna(subset=['AST_1', 'AST_2']).reset_index(drop=True)

    def get_sequence_similarity(str1, str2):
        return difflib.SequenceMatcher(None, str1, str2).ratio()
    
    df['seq_similarity'] = df.apply(lambda row: get_sequence_similarity(row['AST_1'], row['AST_2']), axis=1)

    def get_length_ratio(str1, str2):
        len1, len2 = len(str1), len(str2)
        if max(len1, len2) == 0: return 0
        return min(len1, len2) / max(len1, len2)
    
    df['length_ratio'] = df.apply(lambda row: get_length_ratio(row['AST_1'], row['AST_2']), axis=1)

    vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 3))

    all_asts = df['AST_1'].tolist() + df['AST_2'].tolist()
    vectorizer.fit(all_asts)

    def get_cosine_sim(row):
        vecs = vectorizer.transform([row['AST_1'], row['AST_2']])
        return cosine_similarity(vecs[0], vecs[1])[0][0]

    df['cosine_similarity'] = df.apply(get_cosine_sim, axis=1)

    def get_jaccard_similarity(s1, s2):
        nodes1 = set(re.findall(r'([A-Z][a-zA-Z0-9_]*)\(', s1))
        nodes2 = set(re.findall(r'([A-Z][a-zA-Z0-9_]*)\(', s2))
        if not nodes1 and not nodes2: return 1.0
        return len(nodes1.intersection(nodes2)) / len(nodes1.union(nodes2))
        
    df['jaccard_similarity'] = df.apply(lambda r: get_jaccard_similarity(r['AST_1'], r['AST_2']), axis=1)


    def get_max_depth(ast_string):
        current_depth = max_depth = 0
        for char in ast_string:
            if char in '([':
                current_depth += 1
                if current_depth > max_depth: max_depth = current_depth
            elif char in ')]':
                current_depth -= 1
        return max_depth

    df['depth_difference'] = df.apply(lambda r: abs(get_max_depth(r['AST_1']) - get_max_depth(r['AST_2'])), axis=1)


    feature_cols = [
        'seq_similarity', 'length_ratio', 'cosine_similarity', 
        'jaccard_similarity', 'depth_difference'
    ]
    
    final_cols = ['File_1', 'File_2'] + feature_cols + ['Label']
    df_final = df[final_cols]

    df_final.to_csv("Data/fixed_dataset.csv", index=False)

    print(df.head())

In [12]:
if __name__ == "__main__":
    extract_features()

           File_1          File_2  Label  \
0  submission1.py  submission2.py      1   
1  submission1.py  submission3.py      0   
2  submission1.py  submission4.py      1   
3  submission1.py  submission5.py      0   
4  submission2.py  submission3.py      0   

                                               AST_1  \
0  Module([FunctionDef('func_1', arguments([], [a...   
1  Module([FunctionDef('func_1', arguments([], [a...   
2  Module([FunctionDef('func_1', arguments([], [a...   
3  Module([FunctionDef('func_1', arguments([], [a...   
4  Module([FunctionDef('func_1', arguments([], [a...   

                                               AST_2  seq_similarity  \
0  Module([FunctionDef('func_1', arguments([], [a...        1.000000   
1  Module([FunctionDef('func_1', arguments([], [a...        0.978261   
2  Module([FunctionDef('func_1', arguments([], [a...        0.786787   
3  Module([FunctionDef('func_1', arguments([], [a...        0.974729   
4  Module([FunctionDef('func_1', argum